# 02. Plan-Act 与 Tool Use

ReAct 是「边想边做」，Plan-Act 是「先想清楚再做」。两种模式有各自的适用场景。

- **Plan-Act** 适合可预测任务：步骤已知、依赖关系清晰（如「部署一个服务」）。
- **ReAct** 适合探索性任务：环境不确定、需要边走边调整（如「调试一个 bug」）。

> **检查点**：能用一句话描述 Plan-Act 与 ReAct 的核心取舍。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
# Plan-Act 模式：先规划任务列表，再逐步执行
from dataclasses import dataclass, field

@dataclass
class Task:
    id: str
    desc: str
    depends_on: list = field(default_factory=list)
    done: bool = False

def plan(goal: str) -> list[Task]:
    """任务分解"""
    if "deploy" in goal.lower():
        return [
            Task("T1", "Check environment"),
            Task("T2", "Build artifacts", depends_on=["T1"]),
            Task("T3", "Run tests", depends_on=["T2"]),
            Task("T4", "Deploy to production", depends_on=["T3"]),
        ]
    return [Task("T1", f"Analyze: {goal}"), Task("T2", "Execute", depends_on=["T1"])]

tasks = plan("deploy new version")
completed = set()

for task in tasks:
    # 检查依赖
    missing = set(task.depends_on) - completed
    if missing:
        print(f"{task.id}: SKIP — waiting for {missing}")
        continue
    task.done = True
    completed.add(task.id)
    print(f"{task.id}: {task.desc} ✓")

print(f"\nCompleted: {len(completed)}/{len(tasks)}")

T1: Check environment ✓
T2: Build artifacts ✓
T3: Run tests ✓
T4: Deploy to production ✓

Completed: 4/4


## Tool Use / Function Calling

Agent 的「手」是工具。每个工具需要三个要素：

1. **Name** — 唯一标识
2. **Schema** — 参数的类型和约束（JSON Schema）
3. **Implementation** — 实际执行逻辑

> **检查点**：能写出一个工具的完整定义（name + inputSchema + 实现）。

In [3]:
# Tool 的完整契约
import json

class ToolRegistry:
    """工具注册表：管理 name → schema → implementation 的映射"""
    
    def __init__(self):
        self._tools = {}
    
    def register(self, name: str, schema: dict, fn):
        self._tools[name] = {"schema": schema, "fn": fn}
    
    def list_tools(self) -> list[dict]:
        return [{"name": n, **t["schema"]} for n, t in self._tools.items()]
    
    def call(self, name: str, args: dict) -> str:
        if name not in self._tools:
            return f"Error: tool '{name}' not found"
        return self._tools[name]["fn"](**args)

registry = ToolRegistry()

# 注册 calculator 工具
registry.register(
    "calculator",
    {"description": "Evaluate math expression",
     "parameters": {"expr": {"type": "string", "description": "Expression to evaluate"}}},
    lambda expr: str(eval(expr, {"__builtins__": {}}, {}))
)

# 注册 weather 工具
registry.register(
    "weather",
    {"description": "Get city weather",
     "parameters": {"city": {"type": "string"}}},
    lambda city: f"{city}: Sunny, 25°C"
)

print("Available tools:")
for t in registry.list_tools():
    print(f"  {t['name']}: {t['description']}")

print(f"\ncalculator('2+3*4') = {registry.call('calculator', {'expr': '2+3*4'})}")
print(f"weather('Beijing') = {registry.call('weather', {'city': 'Beijing'})}")
print(f"unknown_tool() = {registry.call('hack', {})}")

Available tools:
  calculator: Evaluate math expression
  weather: Get city weather

calculator('2+3*4') = 14
weather('Beijing') = Beijing: Sunny, 25°C
unknown_tool() = Error: tool 'hack' not found


## 真实 API 实验

真实 function calling：把工具 schema 交给模型，由它决定调用并回填结果（OpenAI tools 协议，MCP 的底层同源机制）。


In [4]:
# 真实 function calling 循环
tools = [
    {"type": "function", "function": {"name": "calculator", "description": "计算数学表达式", "parameters": {"type": "object", "properties": {"expr": {"type": "string"}}, "required": ["expr"]}}},
    {"type": "function", "function": {"name": "search", "description": "搜索知识库", "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}},
]

def calculator(args):
    return str(eval(args["expr"], {"__builtins__": {}}, {}))

def search(args):
    return f"[知识库] 关于 {args['query']}：ReAct 由 Yao 等人 2022 年提出。"

messages = [{"role": "user", "content": "ReAct 是哪一年提出的？提出的年份距今(2026)多少年？算一下。"}]
final, trace = chat_with_tools(client, messages, tools, {"calculator": calculator, "search": search}, verbose=True)
show_trace(trace, "真实工具调用")
print("\n最终回答:", final)


  🔧 search({'query': 'ReAct 提出 年份'}) → [知识库] 关于 ReAct 提出 年份：ReAct 由 Yao 等人 2022 年提出。


=== 真实工具调用 (1 calls) ===
[1] search({'query': 'ReAct 提出 年份'})
    → [知识库] 关于 ReAct 提出 年份：ReAct 由 Yao 等人 2022 年提出。

最终回答: ReAct 是由 **Yao 等人于 2022 年**提出的。

计算距今（2026 年）的年数：

\[
2026 - 2022 = 4 \text{ 年}
\]

**答案：** ReAct 于 **2022 年**提出，距今（2026 年）已有 **4 年**。


## Plan-Act vs ReAct 选择指南

```
任务可预测？
  ├── YES → Plan-Act（先分解，再执行）
  └── NO  → 需要探索？
              ├── YES → ReAct（边想边做）
              └── NO  → 混合（Plan-Act with re-plan on failure）
```

## 练习

1. 在 `ToolRegistry` 中新增一个 `file_reader` 工具：接受 `path` 参数，返回文件内容。
2. 实现一个混合 Agent：先用 Plan-Act 分解任务，每个步骤内部用 ReAct 循环执行。
3. 思考 Tool Schema 的版本管理问题：如果 `calculator` 从接受 `expr` 变成接受 `expression`，旧 Agent 会发生什么？

> **检查点**：能用 ToolRegistry 模式为你的 Agent 注册 3 个工具，并通过 JSON Schema 让其他开发者理解每个工具的接口。